# X5_P1 — Tabla maestra tabulada en el tiempo

Primer script de **X5_alt** (versión paralela y simplificada de X5 — ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md), no reemplaza
`X5_macro_brain.py`). El objetivo de este notebook es construir una **tabla
continua en el tiempo** (una fila por vela H1), integrando solo tres fuentes:

- **Precio** (`Data/{valor}.csv`) — OHLCV nativo H1.
- **X3** (`X3_technical_features.py`) — indicadores técnicos, calculados sobre
  el histórico completo de precio.
- **X2** (`resources/x2/x2_history.json`) — score fundamental, con
  *forward-fill* (los registros son escasos: ver Sección 3).

**Fuera de v0** (ver plan §5): distancia a soportes (requiere reconstruir la
serie histórica de soportes, trabajo aparte) y los parámetros de configuración
(K, N_EXP, LAMBDA, A, B — son constantes en `config.py`, no aportan a un
análisis de correlación temporal).

**Input único que se puede cambiar:**

In [1]:
valor = 'BTCUSD'  # inicialmente solo BTCUSD (ver plan §3)

## 0. Preparación

Importamos `pandas`/`numpy` y reutilizamos `_calcular_todos_indicadores` de
`X3_technical_features.py` en vez de reimplementar los indicadores — así
cualquier cambio futuro en X3 se refleja automáticamente acá.

In [2]:
import json  # para leer x2_history.json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402
from X3_technical_features import _calcular_todos_indicadores  # noqa: E402  (reusa X3 en vez de reimplementar indicadores)

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activo: {valor}")

Activo: BTCUSD


## 1. Cargar precio

`Data/{valor}.csv` no viene ordenado por `DateTime` (ver plan §4) — hay que
ordenar y eliminar duplicados antes de usarlo.

In [3]:
path_precio = cfg.CARPETA_DATA / f'{valor}.csv'
df_precio = pd.read_csv(path_precio)
df_precio['DateTime'] = pd.to_datetime(df_precio['DateTime'])

# El CSV no viene ordenado por fecha y puede traer velas duplicadas —
# hay que resolver ambas cosas antes de calcular cualquier indicador (X3 asume orden).
df_precio = (df_precio.sort_values('DateTime')
                       .drop_duplicates(subset=['DateTime'])
                       .reset_index(drop=True))

print(f"{len(df_precio)} velas H1 — {df_precio['DateTime'].min()} → {df_precio['DateTime'].max()}")
df_precio.head()

40283 velas H1 — 2021-10-27 16:00:00 → 2026-06-02 21:00:00


,DateTime,Open,High,Low,Close,Tick_Volume,Spread,Real_Volume
0,2021-10-27 16:00:00,58734.06,59042.92,58532.74,58856.56,1347.0,586,0.0
1,2021-10-27 17:00:00,58856.56,59166.83,58831.57,58942.47,2065.0,1402,0.0
2,2021-10-28 09:00:00,60431.05,61276.66,60360.70,61048.85,2631.0,3977,0.0
3,2021-10-28 10:00:00,61048.79,61231.67,60814.46,61042.45,4085.0,3349,0.0
4,2021-10-28 11:00:00,61041.32,61106.04,60732.14,60999.73,3942.0,681,0.0


## 2. Indicadores técnicos (X3)

Se calculan sobre el histórico completo (no incremental como en producción)
para que los indicadores con warm-up largo (EMA, RSI, SMA 200) sean correctos
desde la primera vela con datos suficientes.

Se descartan las columnas de distancia a soportes (`dist_nearest_support`,
`dist_floor_support`, `density_2pct`) — fuera de v0, ver Sección 0.

In [4]:
# conjunto_N vacío = sin distancia a soportes (fuera de v0, ver Sección 0);
# con set() las 3 columnas dist_*/density_2pct salen en NaN, así que se descartan abajo.
df_x3 = _calcular_todos_indicadores(df_precio, set())
df_x3 = df_x3.drop(columns=['dist_nearest_support', 'dist_floor_support', 'density_2pct'])

# Prefijo x3_ para distinguir estas columnas de las de precio y X2 en la tabla final
df_x3 = df_x3.rename(columns={c: f'x3_{c}' for c in df_x3.columns if c != 'datetime'})

print(f"{df_x3.shape[1] - 1} features técnicas (X3)")
df_x3.head()

26 features técnicas (X3)


,datetime,x3_sma_20,x3_sma_dist_20,x3_sma_50,x3_sma_dist_50,x3_sma_200,x3_sma_dist_200,x3_ema_12,x3_ema_dist_12,x3_ema_26,...,x3_bb_width,x3_bb_pos,x3_roc_10,x3_roc_20,x3_vol_24h,x3_vol_7d,x3_drawdown_20,x3_drawdown_50,x3_trend_slope_20,x3_trend_slope_50
0,2021-10-27 16:00:00,NaN,NaN,NaN,NaN,NaN,NaN,58856.560000,0.000000,58856.560000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-10-27 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,58869.776923,0.001233,58862.923704,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-10-28 09:00:00,NaN,NaN,NaN,NaN,NaN,NaN,59205.018935,0.030203,59024.844170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-10-28 10:00:00,NaN,NaN,NaN,NaN,NaN,NaN,59487.700637,0.025470,59174.296454,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-10-28 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,59720.320539,0.020974,59309.513754,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Fundamentales (X2)

`x2_history.json` es un registro por (fecha, activo) — no una serie H1. Se
filtra por `{valor}` y más adelante (Sección 4) se pega a la tabla H1 con
`merge_asof` en dirección `backward` (forward-fill: cada vela hereda el último
score conocido). Para BTCUSD hoy hay solo 2 registros históricos — el score
quedará constante en la enorme mayoría del rango (limitación real de datos,
no un bug, ver plan §4).

In [5]:
path_x2 = cfg.CARPETA_FUNDAMENTALS / 'x2_history.json'
with open(path_x2) as f:
    x2_hist = json.load(f)

# x2_history.json trae todos los activos mezclados — nos quedamos solo con {valor}
registros = [e for e in x2_hist if e.get('activo') == valor]

# Solo los 3 scores de nivel superior, no los 'components' en bruto (parsimonia,
# ver principios heredados en el plan §2); se puede sumar detalle más adelante si hace falta.
df_x2 = pd.DataFrame([
    {
        'DateTime': pd.to_datetime(e['date']),
        'x2_score': e.get('score'),
        'x2_score_cross': e.get('score_cross'),
        'x2_score_tendencia': e.get('score_tendencia'),
    }
    for e in registros
]).sort_values('DateTime').reset_index(drop=True)  # merge_asof exige orden ascendente en ambos lados

print(f"{len(df_x2)} registros de X2 para {valor}")
df_x2

2 registros de X2 para BTCUSD


,DateTime,x2_score,x2_score_cross,x2_score_tendencia
0,2026-06-12,0.4906,0.4882,0.5
1,2026-06-14,0.4990,0.4988,0.5


## 4. Ensamblar tabla maestra

Una fila por vela H1: precio (OHLCV) + `x3_*` + `x2_*`.

In [6]:
# Precio y X3 comparten índice/orden (X3 se calculó sobre df_precio) → join exacto por DateTime
tabla = df_precio.merge(df_x3, left_on='DateTime', right_on='datetime').drop(columns=['datetime'])

# merge_asof(direction='backward') = forward-fill: cada vela hereda el último
# registro de X2 con fecha <= la suya (NaN si todavía no existía ningún registro)
tabla = pd.merge_asof(tabla, df_x2, on='DateTime', direction='backward')

print(f"Tabla maestra: {tabla.shape[0]} filas x {tabla.shape[1]} columnas")
tabla.head()

Tabla maestra: 40283 filas x 37 columnas


,DateTime,Open,High,Low,Close,Tick_Volume,Spread,Real_Volume,x3_sma_20,x3_sma_dist_20,...,x3_roc_20,x3_vol_24h,x3_vol_7d,x3_drawdown_20,x3_drawdown_50,x3_trend_slope_20,x3_trend_slope_50,x2_score,x2_score_cross,x2_score_tendencia
0,2021-10-27 16:00:00,58734.06,59042.92,58532.74,58856.56,1347.0,586,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-10-27 17:00:00,58856.56,59166.83,58831.57,58942.47,2065.0,1402,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-10-28 09:00:00,60431.05,61276.66,60360.70,61048.85,2631.0,3977,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-10-28 10:00:00,61048.79,61231.67,60814.46,61042.45,4085.0,3349,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-10-28 11:00:00,61041.32,61106.04,60732.14,60999.73,3942.0,681,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Guardar tabla final

A `resources/x5_alt/{valor}_tabla_maestra.csv`, para que `X5_P2.py` la consuma
sin recalcular.

In [7]:
cfg.CARPETA_X5_ALT.mkdir(parents=True, exist_ok=True)  # resources/x5_alt/ no existe en un checkout nuevo
path_out = cfg.CARPETA_X5_ALT / f'{valor}_tabla_maestra.csv'
tabla.to_csv(path_out, index=False)

print(f"Guardado: {path_out}")

Guardado: /Users/macasaez/Desktop/claude_projects_v2/Alginvesting/resources/x5_alt/BTCUSD_tabla_maestra.csv


## 6. Vista rápida

In [8]:
print(f"Rango: {tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

# Si el precio local (Mac) termina antes de la fecha del primer registro de X2,
# el forward-fill no tiene nada que propagar hacia atrás y x2_score queda NaN en todo el rango.
print(f"Filas sin x2_score (antes del primer registro de X2): {tabla['x2_score'].isna().sum()} / {len(tabla)}")
tabla.tail()

Rango: 2021-10-27 16:00:00 → 2026-06-02 21:00:00
Filas sin x2_score (antes del primer registro de X2): 40283 / 40283


,DateTime,Open,High,Low,Close,Tick_Volume,Spread,Real_Volume,x3_sma_20,x3_sma_dist_20,...,x3_roc_20,x3_vol_24h,x3_vol_7d,x3_drawdown_20,x3_drawdown_50,x3_trend_slope_20,x3_trend_slope_50,x2_score,x2_score_cross,x2_score_tendencia
40278,2026-06-02 17:00:00,67312.89,67738.16,67248.94,67376.67,4442.0,980,0.0,69608.4100,-0.033123,...,-0.050383,0.412205,0.319162,-0.055280,-0.087582,-0.065295,-0.092784,NaN,NaN,NaN
40279,2026-06-02 18:00:00,67378.13,67555.32,66995.90,67220.19,3893.0,980,0.0,69404.0475,-0.032488,...,-0.057319,0.412103,0.319347,-0.057474,-0.089701,-0.067616,-0.096810,NaN,NaN,NaN
40280,2026-06-02 19:00:00,67223.58,67406.18,66318.83,67218.93,6911.0,980,0.0,69199.0335,-0.029458,...,-0.057492,0.408286,0.318499,-0.055828,-0.089718,-0.067908,-0.099862,NaN,NaN,NaN
40281,2026-06-02 20:00:00,67215.49,67586.39,66801.78,67484.09,5636.0,980,0.0,69013.5615,-0.022664,...,-0.052104,0.426751,0.319977,-0.047310,-0.086128,-0.065568,-0.101789,NaN,NaN,NaN
40282,2026-06-02 21:00:00,67492.44,67833.48,67307.34,67562.83,3499.0,980,0.0,68863.5525,-0.019252,...,-0.042518,0.425986,0.319894,-0.046199,-0.085061,-0.064385,-0.103527,NaN,NaN,NaN
